# Proyección 2018

In [ ]:
from common import load_artifact, md
# Subtítulo con el nombre del mejor modelo
md(f"*Ventas esperadas por mes con {load_artifact('resumen.json')['mejor_modelo']}, reentrenado con 2015–2017*")

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from common import NEG, POS, PROY, REAL, kpis, load_artifact, load_mensual, md, show, style_fig, tabla, usd

# Opciones de los filtros y nombres de los meses
TODAS = "Todas"
MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
CATS = ["Furniture", "Office Supplies", "Technology"]
REGS = ["Central", "East", "South", "West"]
# Resultados del modelo: resumen, ventas reales, proyección 2018 y prueba de 2017 a 12 meses
resumen = load_artifact("resumen.json")
mensual = load_mensual()
p18 = load_artifact("proyeccion_2018.csv")
rec = load_artifact("prueba_recursiva_2017.csv")


# Filtra por categoría y región ("Todas" deja todas las filas)
def filtrar(df, cat, reg):
    if cat != TODAS:
        df = df[df.Category == cat]
    if reg != TODAS:
        df = df[df.Region == reg]
    return df


def error_esperado(cat, reg):
    """Error de la proyección a 12 meses para este filtro, medido al proyectar 2017 con datos hasta 2016."""
    t = filtrar(rec, cat, reg).groupby("Mes")[["Sales", "Proyeccion"]].sum()
    return float(np.mean(np.abs(t.Sales - t.Proyeccion)) / t.Sales.mean() * 100)


def seleccion(cat, reg):
    """Serie real (2015–2017), proyección 2018 y error típico de una categoría × región."""
    real = filtrar(mensual, cat, reg)
    real = real[real.Mes.dt.year >= 2015].groupby("Mes", as_index=False).Sales.sum()
    proy = filtrar(p18, cat, reg).groupby("Mes", as_index=False).Proyeccion.sum()
    return real, proy, error_esperado(cat, reg)


# Cifras del total de la empresa
real, proy, err = seleccion(TODAS, TODAS)
v17, v18 = real[real.Mes.dt.year == 2017].Sales.sum(), proy.Proyeccion.sum()
# Mes con la mayor venta proyectada
pico = proy.loc[proy.Proyeccion.idxmax()]
# Cifras clave de la proyección
kpis([
    ("Proyección 2018", usd(v18), f"real 2017: {usd(v17)}"),
    ("Variación esperada", f"{(v18 / v17 - 1) * 100:+.1f} %", "frente a 2017"),
    ("Mes pico", MESES[pico.Mes.month - 1].capitalize(), usd(pico.Proyeccion)),
    ("Error típico mensual", f"±{err:.0f} %", "proyección a 12 meses (prueba 2017)"),
])

Cifras del total de la empresa. En la gráfica, el menú permite ver una categoría, una región o una
combinación de las dos.

## Ventas mensuales: real 2015–2017 y proyección 2018

In [ ]:
# Todas las vistas del menú: empresa, cada categoría, cada región y cada combinación
opciones = ([(TODAS, TODAS)] + [(c, TODAS) for c in CATS] + [(TODAS, r) for r in REGS]
            + [(c, r) for c in CATS for r in REGS])
fig = go.Figure()
botones = []
# Tres trazas por vista (banda de error, real y proyección); solo la primera vista es visible
for i, (cat, reg) in enumerate(opciones):
    real, proy, err = seleccion(cat, reg)
    # Banda: proyección ± el error típico medido al proyectar 2017
    lo, hi = proy.Proyeccion * (1 - err / 100), proy.Proyeccion * (1 + err / 100)
    # La línea de la proyección arranca en el último mes real para que las dos curvas queden unidas
    p2 = pd.concat([real.tail(1).rename(columns={"Sales": "Proyeccion"}), proy])
    vis = i == 0
    fig.add_trace(go.Scatter(x=list(proy.Mes) + list(proy.Mes[::-1]), y=list(hi) + list(lo[::-1]),
                             fill="toself", fillcolor="rgba(224,150,43,0.18)", line_width=0, hoverinfo="skip",
                             name=f"Error típico (±{err:.0f} %)", visible=vis))
    fig.add_trace(go.Scatter(x=real.Mes, y=real.Sales, name="Real", mode="lines+markers",
                             line=dict(color=REAL, width=2), marker_size=4, visible=vis))
    fig.add_trace(go.Scatter(x=p2.Mes, y=p2.Proyeccion, name="Proyección 2018", mode="lines+markers",
                             line=dict(color=PROY, width=2.5, dash="dash"), marker_size=5, visible=vis))
    # Nombre de la vista en el menú
    if cat == TODAS and reg == TODAS:
        etiqueta = "Toda la empresa"
    elif reg == TODAS:
        etiqueta = cat
    elif cat == TODAS:
        etiqueta = f"Región {reg}"
    else:
        etiqueta = f"{cat} · {reg}"
    # Cada botón muestra las 3 trazas de su vista y oculta las demás
    botones.append(dict(label=etiqueta, method="update",
                        args=[{"visible": [j // 3 == i for j in range(3 * len(opciones))]}]))
fig.update_layout(updatemenus=[dict(buttons=botones, active=0, x=0, xanchor="left", y=1.18, yanchor="top")])
fig.update_yaxes(title="Ventas del mes (USD)", tickprefix="$", tickformat=",.0f")
style_fig(fig, 420)
fig.update_layout(margin=dict(t=60))
show(fig)

La banda muestra el error típico de una proyección a 12 meses para esta selección, medido al proyectar
2017 sin conocerlo. Para el total de la empresa el error es menor que para una sola categoría en una
región, porque al sumar series los errores se compensan.

## Proyección por categoría y región

### 2018 frente a 2017

In [ ]:
# Ventas reales 2017 y proyección 2018 por categoría × región
r17 = mensual[mensual.Mes.dt.year == 2017].groupby(["Category", "Region"]).Sales.sum()
pr = p18.groupby(["Category", "Region"]).Proyeccion.sum()
t = r17.to_frame("v2017").join(pr.rename("v2018")).reset_index()
# Variación esperada en %
t["crec"] = (t.v2018 / t.v2017 - 1) * 100
t = t.sort_values("v2018", ascending=False)


# Rojo para caídas, verde para crecimientos de 15 % o más
def color(v):
    return f"color: {NEG}; font-weight: 700" if v < 0 else (f"color: {POS}; font-weight: 700" if v >= 15 else "")


# Tabla con formato en dólares y porcentaje
(tabla(t.rename(columns={"Category": "Categoría", "Region": "Región", "v2017": "Ventas 2017",
                         "v2018": "Proyección 2018", "crec": "Variación"}))
   .format({"Ventas 2017": usd, "Proyección 2018": usd, "Variación": "{:+.1f} %"})
   .map(color, subset=["Variación"]))

En rojo, las combinaciones donde se proyecta una caída; en verde, crecimientos de 15 % o más.

### Lectura para el negocio

In [ ]:
# Series con caída proyectada y la de mayor crecimiento
caidas = t[t.crec < 0]
top = t.sort_values("crec", ascending=False).iloc[0]
# Peso de septiembre a diciembre en el total proyectado del año
temporada = p18[p18.Mes.dt.month >= 9].Proyeccion.sum() / p18.Proyeccion.sum() * 100
# Texto con las series que caen (o un punto si no hay ninguna)
lista_caidas = (": " + ", ".join(f"{c.Category} en {c.Region} ({c.crec:+.0f} %)" for c in caidas.itertuples())
                if len(caidas) else ".")
# Lectura de la proyección escrita con las cifras calculadas
md(f"- Se proyectan {usd(resumen['proyeccion_2018'])} en 2018, "
   f"{(resumen['proyeccion_2018'] / resumen['ventas_2017'] - 1) * 100:+.0f} % frente a 2017 "
   f"({usd(resumen['ventas_2017'])}). El modelo extiende el crecimiento de 2016–2017.\n"
   f"- La temporada alta (septiembre a diciembre) concentra el {temporada:.0f} % de las ventas del año en "
   f"solo 4 meses: es el periodo para asegurar inventario y personal.\n"
   f"- Mayor crecimiento esperado: {top.Category} en {top.Region} ({top.crec:+.0f} %).\n"
   f"- Combinaciones con caída proyectada: {len(caidas)}{lista_caidas}\n"
   f"- Usar la proyección como punto de partida para metas e inventario, ajustado con lo que el modelo no "
   f"conoce: campañas, precios y competencia.")